<a href="https://colab.research.google.com/github/nexageapps/AI/blob/main/Basic%20%7C%20L7%20-%20Convolutional%20Neural%20Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 7: Convolutional Neural Networks (CNNs)

**Author:** Karthik Arjun  
**LinkedIn:** [karthik-arjun-a5b4a258](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Created:** 2026  
**Updated:** 2026

## Learning Objectives
- Understand why CNNs are superior for image tasks
- Learn convolution operations and filters
- Understand pooling layers (MaxPooling, AveragePooling)
- Build a CNN for image classification
- Explore transfer learning basics

## Prerequisites
- Completed L1-L6
- Understanding of neural networks and activation functions

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
import seaborn as sns

print(f"TensorFlow version: {tf.__version__}")

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

## 1. Why CNNs for Images?

### Problems with Fully Connected Networks:
1. **Too many parameters:** 28x28 image = 784 inputs → millions of parameters
2. **No spatial awareness:** Treats pixels independently
3. **Not translation invariant:** Cat in top-left ≠ cat in bottom-right

### CNN Advantages:
1. **Parameter sharing:** Same filter applied across entire image
2. **Spatial hierarchy:** Learns local patterns first, then combines them
3. **Translation invariant:** Detects features regardless of position

## 2. Understanding Convolution Operation

A **convolution** slides a small filter (kernel) over the image.

**Filter/Kernel:** Small matrix (e.g., 3x3) that detects specific features
- Edge detection
- Corner detection
- Texture patterns

In [ ]:
# Demonstrate convolution with edge detection
from scipy import signal

# Create a simple image
image = np.zeros((10, 10))
image[3:7, 3:7] = 1  # White square in center

# Define edge detection filters
vertical_edge = np.array([[-1, 0, 1],
                          [-1, 0, 1],
                          [-1, 0, 1]])

horizontal_edge = np.array([[-1, -1, -1],
                            [ 0,  0,  0],
                            [ 1,  1,  1]])

# Apply convolution
vertical_edges = signal.convolve2d(image, vertical_edge, mode='same')
horizontal_edges = signal.convolve2d(image, horizontal_edge, mode='same')

# Visualize
plt.figure(figsize=(15, 4))

plt.subplot(1, 4, 1)
plt.imshow(image, cmap='gray')
plt.title('Original Image', fontsize=12)
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(vertical_edge, cmap='gray')
plt.title('Vertical Edge Filter', fontsize=12)
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(vertical_edges, cmap='gray')
plt.title('Vertical Edges Detected', fontsize=12)
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(horizontal_edges, cmap='gray')
plt.title('Horizontal Edges Detected', fontsize=12)
plt.axis('off')

plt.tight_layout()
plt.show()

print("Convolution Operation:")
print("1. Slide filter over image")
print("2. Multiply overlapping values")
print("3. Sum the results")
print("4. Move to next position")

## 3. Understanding Pooling

**Pooling** reduces spatial dimensions while retaining important information.

### Types:
- **MaxPooling:** Takes maximum value in each region
- **AveragePooling:** Takes average value in each region

### Benefits:
- Reduces parameters
- Provides translation invariance
- Reduces overfitting

In [ ]:
# Demonstrate pooling
sample_image = np.array([[1, 3, 2, 4],
                         [5, 6, 1, 3],
                         [2, 1, 4, 2],
                         [3, 2, 3, 1]])

# Max pooling (2x2)
max_pooled = np.array([[6, 4],
                       [3, 4]])

# Average pooling (2x2)
avg_pooled = np.array([[3.75, 2.5],
                       [2.0, 2.5]])

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(sample_image, cmap='viridis')
plt.title('Original (4x4)', fontsize=12)
for i in range(4):
    for j in range(4):
        plt.text(j, i, str(sample_image[i, j]), 
                ha='center', va='center', color='white', fontsize=14)
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(max_pooled, cmap='viridis')
plt.title('Max Pooling (2x2)', fontsize=12)
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(max_pooled[i, j]), 
                ha='center', va='center', color='white', fontsize=14)
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(avg_pooled, cmap='viridis')
plt.title('Average Pooling (2x2)', fontsize=12)
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{avg_pooled[i, j]:.1f}", 
                ha='center', va='center', color='white', fontsize=14)
plt.axis('off')

plt.tight_layout()
plt.show()

## 4. Load and Prepare CIFAR-10 Dataset

CIFAR-10: 60,000 color images (32x32) in 10 classes
- airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck

In [ ]:
# Load CIFAR-10
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Training data shape: {x_train.shape}")
print(f"Test data shape: {x_test.shape}")
print(f"Image shape: {x_train[0].shape}")
print(f"Number of classes: {len(class_names)}")

# Visualize samples
plt.figure(figsize=(12, 6))
for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]], fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Normalize pixel values
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Convert labels to one-hot
y_train_onehot = keras.utils.to_categorical(y_train, 10)
y_test_onehot = keras.utils.to_categorical(y_test, 10)

print(f"Normalized training data range: [{x_train.min():.2f}, {x_train.max():.2f}]")

## 5. Build a Simple CNN

Architecture:
```
Input (32x32x3)
  ↓
Conv2D (32 filters, 3x3) + ReLU
  ↓
MaxPooling (2x2)
  ↓
Conv2D (64 filters, 3x3) + ReLU
  ↓
MaxPooling (2x2)
  ↓
Flatten
  ↓
Dense (64) + ReLU
  ↓
Dense (10) + Softmax
```

In [ ]:
# Build CNN model
model_cnn = keras.Sequential([
    # First convolutional block
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    
    # Second convolutional block
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Third convolutional block
    layers.Conv2D(64, (3, 3), activation='relu'),
    
    # Flatten and dense layers
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='Simple_CNN')

model_cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_cnn.summary()

## 6. Train the CNN

In [ ]:
# Train the model
print("Training CNN...")
history_cnn = model_cnn.fit(
    x_train, y_train_onehot,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

# Evaluate
test_loss, test_acc = model_cnn.evaluate(x_test, y_test_onehot)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")

In [ ]:
# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_cnn.history['accuracy'], label='Training')
plt.plot(history_cnn.history['val_accuracy'], label='Validation')
plt.title('Model Accuracy', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_cnn.history['loss'], label='Training')
plt.plot(history_cnn.history['val_loss'], label='Validation')
plt.title('Model Loss', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Improved CNN with Dropout and Batch Normalization

In [ ]:
# Build improved CNN
model_improved = keras.Sequential([
    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Dense layers
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
], name='Improved_CNN')

model_improved.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_improved.summary()

In [ ]:
# Train improved model
print("Training improved CNN...")
history_improved = model_improved.fit(
    x_train, y_train_onehot,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

# Evaluate
test_loss, test_acc = model_improved.evaluate(x_test, y_test_onehot)
print(f"\nImproved Model Test Accuracy: {test_acc*100:.2f}%")

## 8. Visualize Predictions

In [ ]:
# Make predictions
predictions = model_improved.predict(x_test[:20])

# Visualize
plt.figure(figsize=(15, 6))
for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(x_test[i])
    
    pred_label = np.argmax(predictions[i])
    true_label = y_test[i][0]
    confidence = predictions[i][pred_label] * 100
    
    color = 'green' if pred_label == true_label else 'red'
    plt.title(f"P:{class_names[pred_label]}\nT:{class_names[true_label]}\n({confidence:.0f}%)", 
              color=color, fontsize=8)
    plt.axis('off')

plt.tight_layout()
plt.show()

## 9. Visualize Learned Filters

Let's see what the first convolutional layer learned!

In [ ]:
# Get first conv layer weights
first_layer = model_improved.layers[0]
filters, biases = first_layer.get_weights()

print(f"Filter shape: {filters.shape}")  # (3, 3, 3, 32)
print(f"Number of filters: {filters.shape[-1]}")

# Normalize filters for visualization
f_min, f_max = filters.min(), filters.max()
filters_normalized = (filters - f_min) / (f_max - f_min)

# Plot first 32 filters
plt.figure(figsize=(12, 6))
for i in range(32):
    plt.subplot(4, 8, i + 1)
    # Take the filter for all input channels
    filter_img = filters_normalized[:, :, :, i]
    plt.imshow(filter_img)
    plt.axis('off')
plt.suptitle('Learned Filters in First Conv Layer', fontsize=14)
plt.tight_layout()
plt.show()

## 10. Key Takeaways

### CNN Architecture:
1. **Convolutional layers** detect local patterns (edges, textures)
2. **Pooling layers** reduce dimensions and provide translation invariance
3. **Multiple conv blocks** learn hierarchical features
4. **Flatten + Dense** layers for final classification

### Best Practices:
5. Use **padding='same'** to maintain spatial dimensions
6. Add **BatchNormalization** after conv layers
7. Use **Dropout** to prevent overfitting
8. Increase filters depth as you go deeper (32 → 64 → 128)

### Performance:
9. CNNs significantly outperform fully connected networks on images
10. Fewer parameters but better accuracy due to parameter sharing

## Next Steps (L8)
- Learn about Recurrent Neural Networks (RNNs)
- Understand sequence modeling
- Implement LSTM/GRU for text generation

## References
- CS231n CNN for Visual Recognition: http://cs231n.stanford.edu/
- Deep Learning Book Chapter 9: https://www.deeplearningbook.org/
- TensorFlow CNN Tutorial: https://www.tensorflow.org/tutorials/images/cnn